# Sprint 0 — Setup & Audit Initial du Dataset

**AquaSense AI · EHTP MIG S4**

Ce notebook valide :
- La présence des fichiers CSV dans `data/raw/`
- Les dimensions du dataset (59 400 × 40 attendues)
- Les types de colonnes et un aperçu des données
- La distribution de la variable cible `status_group`

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print(f"Racine projet : {PROJECT_ROOT}")
print(f"Dossier raw   : {RAW_DIR}")

## 1. Vérification des fichiers

In [ ]:
REQUIRED_FILES = ["train_values.csv", "train_labels.csv", "test_values.csv"]

for fname in REQUIRED_FILES:
    fpath = RAW_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / (1024 * 1024)
        print(f"✅ {fname} — {size_mb:.2f} Mo")
    else:
        print(f"❌ {fname} — MANQUANT (lancer: python scripts/download_data.py)")

## 2. Chargement des données

In [ ]:
train_values = pd.read_csv(RAW_DIR / "train_values.csv")
train_labels = pd.read_csv(RAW_DIR / "train_labels.csv")
test_values = pd.read_csv(RAW_DIR / "test_values.csv")

print("train_values :", train_values.shape)
print("train_labels :", train_labels.shape)
print("test_values  :", test_values.shape)

## 3. Validation des dimensions (critère S0)

In [ ]:
EXPECTED_ROWS = 59_400
EXPECTED_COLS = 40

assert train_values.shape[0] == EXPECTED_ROWS, f"Lignes: {train_values.shape[0]} != {EXPECTED_ROWS}"
assert train_values.shape[1] == EXPECTED_COLS, f"Colonnes: {train_values.shape[1]} != {EXPECTED_COLS}"
assert train_labels.shape[0] == EXPECTED_ROWS

print(f"✅ Dimensions validées : {EXPECTED_ROWS} lignes × {EXPECTED_COLS} colonnes")

## 4. Types et aperçu

In [ ]:
print("=== dtypes ===")
print(train_values.dtypes.value_counts())
print("\n=== head ===")
train_values.head()

## 5. Distribution de la cible

In [ ]:
target_counts = train_labels["status_group"].value_counts()
target_pct = train_labels["status_group"].value_counts(normalize=True) * 100

summary = pd.DataFrame({"count": target_counts, "pct": target_pct.round(1)})
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
target_counts.plot.bar(ax=axes[0], color=["#2ecc71", "#f39c12", "#e74c3c"])
axes[0].set_title("Distribution status_group (counts)")
axes[0].tick_params(axis="x", rotation=15)
target_pct.plot.pie(ax=axes[1], autopct="%.1f%%", colors=["#2ecc71", "#f39c12", "#e74c3c"])
axes[1].set_title("Distribution status_group (%)")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

## 6. Résumé audit initial

| Élément | Valeur attendue | Statut |
|---------|-----------------|--------|
| train_values shape | (59400, 40) | À confirmer ci-dessus |
| Classes cible | 3 | functional / needs repair / non functional |
| Imbalance | ~54% / ~7% / ~39% | À confirmer ci-dessus |